In [2]:
import os
import sys

os.environ['SPARK_VERSION'] = '3.3'
os.environ["JAVA_HOME"] = '/usr/lib/jvm/java-8-openjdk-amd64/'

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [3]:
import os
import sys
from pyspark.sql import SparkSession
import plotly.graph_objects as go
import plotly.offline as pyo
import pydeequ

In [4]:
spark = (
    SparkSession.builder.appName("IcebergLocalDevelopment")
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.5.2,"
        "org.xerial:sqlite-jdbc:3.42.0.0,"
        "com.amazon.deequ:deequ:2.0.11-spark-3.3"
    )
    .config("spark.sql.crossJoin.enabled", "true")
    .config("spark.sql.iceberg.handle-timestamp-without-timezone", "true")
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
    )
    .config("spark.sql.catalog.default", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.default.type", "jdbc")
    # error with both using catalog-impl and type, we use only type
    # .config(
    #     "spark.sql.catalog.default.catalog-impl",
    #     "org.apache.iceberg.jdbc.JdbcCatalog",
    # )
    .config(
        "spark.sql.catalog.default.uri",
        "jdbc:sqlite:/tmp/warehouse/pyiceberg_catalog.db",
    )
    .config(
        "spark.sql.catalog.default.warehouse",
        "hdfs://localhost:9000/user/hive/warehouse",
    )
    .config("spark.sql.catalog.default.jdbc.driver", "org.sqlite.JDBC")
    .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9000")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .config("spark.jars.excludes", pydeequ.f2j_maven_coord)
    .getOrCreate()
)

25/06/17 13:08:10 WARN Utils: Your hostname, rohitkarki resolves to a loopback address: 127.0.1.1; using 10.13.164.166 instead (on interface wlp4s0)
25/06/17 13:08:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/rohitkarki/.local/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/rohitkarki/.ivy2/cache
The jars for the packages stored in: /home/rohitkarki/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.3_2.12 added as a dependency
org.xerial#sqlite-jdbc added as a dependency
com.amazon.deequ#deequ added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f293ae67-dcd1-4db5-bd62-04f563fa6f51;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.3_2.12;1.5.2 in central
	found org.xerial#sqlite-jdbc;3.42.0.0 in central
	found com.amazon.deequ#deequ;2.0.11-spark-3.3 in central
	found org.scala-lang#scala-reflect;2.12.10 in central
	found org.scalanlp#breeze_2.12;1.2 in central
	found org.scalanlp#breeze-macros_2.12;1.2 in central
	found com.github.fommil.netlib#core;1.1.2 in central
	found net.sf.opencsv#opencsv;2.3 in central
	found com.github.wendykierp#JTransforms;3.1 in central
	found pl.edu.icm#JLargeArrays;1.5 in central
	found org.apache.commons#commons-math3;3.2 in central

25/06/17 13:08:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [5]:
# Check current catalog and namespace
spark.sql("SHOW CURRENT NAMESPACE").show()

# List namespaces in the default catalog specifically
spark.sql("SHOW NAMESPACES IN default").show()

# If sales namespace exists, try to use it
spark.sql("USE default.sales")
spark.sql("SHOW TABLES").show()


+-------------+---------+
|      catalog|namespace|
+-------------+---------+
|spark_catalog|  default|
+-------------+---------+

25/06/17 13:08:18 WARN JdbcCatalog: JDBC catalog is initialized without view support. To auto-migrate the database's schema and enable view support, set jdbc.schema-version=V1
+---------+
|namespace|
+---------+
|    sales|
+---------+

+---------+--------------+-----------+
|namespace|     tableName|isTemporary|
+---------+--------------+-----------+
|    sales|       Drivers|      false|
|    sales|ai_job_dataset|      false|
|    sales|  transactions|      false|
|    sales|      vehicles|      false|
+---------+--------------+-----------+



In [11]:
from pyspark.sql import Row
import time
ut = time.time()
product = [
    {'product_id': '00001', 'product_name': 'Heater', 'price': 250, 'category': 'Electronics', 'updated_at': ut},
    {'product_id': '00002', 'product_name': 'Thermostat', 'price': 400, 'category': 'Electronics', 'updated_at': ut},
    {'product_id': '00003', 'product_name': 'Television', 'price': 600, 'category': 'Electronics', 'updated_at': ut},
    {'product_id': '00004', 'product_name': 'Blender', 'price': 100, 'category': 'Electronics', 'updated_at': ut},
    {'product_id': '00005', 'product_name': 'USB charger', 'price': 50, 'category': 'Electronics', 'updated_at': ut}
]
df_products = spark.createDataFrame(Row(**x) for x in product)
df_products.createOrReplaceTempView('tmp')

spark.sql(f"""
CREATE TABLE product USING iceberg AS SELECT * FROM tmp
""")

DataFrame[]

In [12]:
spark.sql("SELECT * FROM product ORDER BY product_id").show()

+----------+------------+-----+-----------+--------------------+
|product_id|product_name|price|   category|          updated_at|
+----------+------------+-----+-----------+--------------------+
|     00001|      Heater|  250|Electronics|1.7501450725800977E9|
|     00002|  Thermostat|  400|Electronics|1.7501450725800977E9|
|     00003|  Television|  600|Electronics|1.7501450725800977E9|
|     00004|     Blender|  100|Electronics|1.7501450725800977E9|
|     00005| USB charger|   50|Electronics|1.7501450725800977E9|
+----------+------------+-----+-----------+--------------------+



In [14]:
from pyspark.sql import Row
import time
from py4j.protocol import Py4JJavaError
import random

def perform_merge_with_retry(max_retries=5, initial_wait=1):
    attempt = 0
    while attempt < max_retries:
        try:
            spark.sql("""
            MERGE INTO product t 
            USING updates s
            ON t.product_id = s.product_id
            WHEN MATCHED THEN 
                UPDATE SET 
                    product_name = s.product_name,
                    price = s.price,
                    category = s.category,
                    updated_at = s.updated_at
            WHEN NOT MATCHED THEN 
                INSERT (product_id, product_name, price, category, updated_at)
                VALUES (s.product_id, s.product_name, s.price, s.category, s.updated_at)
            """)
            return True
        except Py4JJavaError as e:
            if "database is locked" in str(e):
                wait_time = initial_wait * (2 ** attempt) + random.uniform(0, 1)
                print(f"Database locked, retrying in {wait_time:.2f} seconds...")
                time.sleep(wait_time)
                attempt += 1
            else:
                raise e
    raise Exception("Failed to execute merge after maximum retries")

# Create updated products DataFrame
ut = time.time()
updated_products = [
    # Updated products (existing product_ids)
    {'product_id': '00001', 'product_name': 'Smart Heater', 'price': 300, 'category': 'Electronics', 'updated_at': ut},
    {'product_id': '00002', 'product_name': 'Smart Thermostat', 'price': 450, 'category': 'Electronics', 'updated_at': ut},
    # New products (new product_ids)
    {'product_id': '00006', 'product_name': 'Coffee Maker', 'price': 150, 'category': 'Electronics', 'updated_at': ut},
    {'product_id': '00007', 'product_name': 'Microwave', 'price': 200, 'category': 'Electronics', 'updated_at': ut}
]

# Create DataFrame with updates and register temp view
df_updates = spark.createDataFrame(Row(**x) for x in updated_products)
df_updates.createOrReplaceTempView('updates')

# Perform merge operation with retry mechanism
perform_merge_with_retry()

# Verify the results
spark.sql("SELECT * FROM product ORDER BY product_id").show()

+----------+----------------+-----+-----------+--------------------+
|product_id|    product_name|price|   category|          updated_at|
+----------+----------------+-----+-----------+--------------------+
|     00001|    Smart Heater|  300|Electronics| 1.750145115981464E9|
|     00002|Smart Thermostat|  450|Electronics| 1.750145115981464E9|
|     00003|      Television|  600|Electronics|1.7501450725800977E9|
|     00004|         Blender|  100|Electronics|1.7501450725800977E9|
|     00005|     USB charger|   50|Electronics|1.7501450725800977E9|
|     00006|    Coffee Maker|  150|Electronics| 1.750145115981464E9|
|     00007|       Microwave|  200|Electronics| 1.750145115981464E9|
+----------+----------------+-----+-----------+--------------------+



In [ ]:
# # Set default catalog to local for convenience
# spark.sql("USE default")

# # Create sample data
# data = [
#     (1, "electronics", 299.99),
#     (2, "clothing", 79.99),
#     (3, "groceries", 45.50),
#     (4, "electronics", 999.99),
#     (5, "clothing", 120.00),
# ]

# # Define schema
# schema = StructType(
#     [
#         StructField("id", LongType(), True),
#         StructField("category", StringType(), True),
#         StructField("amount", DoubleType(), True),
#     ]
# )

# # Create DataFrame
# df = spark.createDataFrame(data, schema)

# # Write DataFrame to Iceberg table
# df.write.format("iceberg").mode("overwrite").saveAsTable("hello.sales_data_df")

# # Read the data back
# print("Data written via DataFrame API:")
# spark.table("hello.sales_data_df").show()

# # Perform analytics
# print("Category-wise totals:")
# spark.sql(
#     """
# SELECT category, 
#        COUNT(*) as count, 
#        SUM(amount) as total_amount,
#        AVG(amount) as avg_amount
# FROM hello.sales_data_df 
# GROUP BY category 
# ORDER BY total_amount DESC
# """
# ).show()
